# 5 Advanced Visualizations

In [8]:
import pandas as pd
import numpy as np
from ift6758.features.simple_visualizations import games_list

In [9]:
def number_games(season) -> dict:
    # Get all the games dataframe
    df_games = games_list(season, events=False)

    # Keep regular season and playoffs only (game type code in the game id)
    df_games = df_games[df_games["game_type"].isin([2, 3])]

    # Calculate the number of game for each team
    number_games_per_team = {}
    for game in df_games.itertuples():
        homeTeam_id = getattr(game, "homeTeam_abbrev")
        awayTeam_id = getattr(game, "awayTeam_abbrev")
        number_games_per_team[homeTeam_id] = number_games_per_team.get(homeTeam_id, 0) + 1
        number_games_per_team[awayTeam_id] = number_games_per_team.get(awayTeam_id, 0) + 1

    return number_games_per_team

print(number_games(2022))

{'NSH': 82, 'SJS': 82, 'NYR': 89, 'TBL': 88, 'LAK': 88, 'VGK': 104, 'WSH': 82, 'BOS': 89, 'CAR': 97, 'CBJ': 82, 'MTL': 82, 'TOR': 93, 'COL': 89, 'CHI': 82, 'ANA': 82, 'SEA': 96, 'EDM': 94, 'VAN': 82, 'PIT': 82, 'ARI': 82, 'PHI': 82, 'NJD': 94, 'BUF': 82, 'OTT': 82, 'NYI': 88, 'FLA': 103, 'DAL': 101, 'MIN': 88, 'CGY': 82, 'DET': 82, 'WPG': 87, 'STL': 82}


In [ ]:

def shot_data(season):
    """
    Gets the data from json (possibly by calling the distance function) for the season and extracts the shot coordinates to a more usable format
    :param season: 4 digit year of the season start
    :return: a dataframe containing each team's name, number of games, and an array of x,y shot coordinates (1 line per shot, columns x and y)
    """
    # Get each team's number of games
    team_game_number = number_games(season)

    # Link each team_id to its abbrev
    df_games = games_list(season, events=False)
    id_to_abbrev = {}
    for game in df_games.itertuples():
        id_to_abbrev[getattr(game, "homeTeam_id")] = getattr(game, "homeTeam_abbrev")
        id_to_abbrev[getattr(game, "awayTeam_id")] = getattr(game, "awayTeam_abbrev")

    # Get the shots and goals dataframe for the given season
    df_events = games_list(season)

    # Keep regular season and playoffs
    df_shots = df_events[
        df_events["game_type"].isin(["regular season", "playoffs"])
    ].copy()

    # Put all the shots on the same side (use same logic as distance() from simple_visualization.py)
    home_shoots = df_shots["team_id"] == df_shots["homeTeam_id"]
    attacks_left = np.where(
        home_shoots,
        df_shots["homeTeamDefendingSide"] == "right",
        df_shots["homeTeamDefendingSide"] == "left"
    )

    # Flip the shots by 180°­­­­ if they were taken from the left side of the pitch
    df_shots.loc[attacks_left, ["x_coord", "y_coord"]] *= -1

    rows = []
    for team_id, df_team in df_shots.groupby("team_id"):
        team = id_to_abbrev[team_id]
        rows.append({
            "team": team,
            "games": team_game_number[team],
            "shots": df_team[["x_coord", "y_coord"]].to_numpy()
        })

    return pd.DataFrame(rows)

print(shot_data(2022).head())

  team  games                                              shots
0  NJD     94  [[75.0, 11.0], [62.0, 24.0], [80.0, -18.0], [-...
1  NYI     88  [[74.0, 1.0], [-90.0, -21.0], [69.0, 24.0], [4...
2  NYR     89  [[54.0, -6.0], [42.0, 19.0], [81.0, -8.0], [53...
3  PHI     82  [[81.0, -2.0], [86.0, 25.0], [75.0, -5.0], [37...
4  PIT     82  [[54.0, -22.0], [80.0, 4.0], [79.0, 15.0], [78...


In [ ]:

def kde():
    """
    Calculates kde when given an array of shot coordinates
    :return: shot density
    """



In [ ]:

class ShotMap:

    def __init__(self, season):
        """

        :param season: 4 digit year of the season start
        """
        self.season = season
        self.shots = shot_data(season)

    def calculate_kdes(self):
        """
        This calculates league average shot rate per hour (league_sph) and excess shot rate per hour for each team, in the form of shot density using kde. Also saves raw difference in shots per team hour
        :return:
        """
        # calculate shot rate per hour density for each team using kde:
        # calculate league average shot rate per hour (league_sph) by averaging over sph per team
        # convert teams' sph to excess sph using league average
        # calculate raw difference in shots per team hour


In [ ]:
shots2020 = ShotMap(2020)
shots2020.calculate_kde()
shots2020.shots.head()

In [ ]:
def interactive_shot_map(shot_map):
    """
    :param shot_map: ShotMap object
    :return: Creates html interactive figure ?
    """
    # checks if shot_map has kdes, if not, calls shot_map.calculate_kde()

In [ ]:
interactive_shot_map(shots2020)